# CS5720 Neural Networks and Deep Learning — Home Assignment 3

**Student name:** Sai Kodali
**University of Central Missouri — Fall 2026**

This notebook has both parts of the assignment:
- **Part I** – short answer questions (written in markdown)
- **Part II** – programming questions (convolution from scratch + transfer learning freeze vs fine-tune)

---
# Part I — Short-Answer Questions

## Question 1 — Convolution, Padding, and Stride

Input image = 32 × 32 × 3, 16 filters of size 5 × 5, stride 1, padding 0.

**a. What are N, F, S and P?**
- N = 32 (input height/width)
- F = 5 (filter size)
- S = 1 (stride)
- P = 0 (padding)

**b. Output height and width**

$$N_{out} = \frac{N - F + 2P}{S} + 1 = \frac{32 - 5 + 0}{1} + 1 = 28$$

So the output feature map is **28 × 28**.

**c. Complete output volume**

Each filter gives one output channel, and there are 16 filters, so the output volume is **28 × 28 × 16**.
(Each filter is actually 5 × 5 × 3 because it goes through all 3 input channels, but it still produces just one 2D map.)

**d. Padding to keep 32 × 32**

We need N_out = N:

$$32 = \frac{32 - 5 + 2P}{1} + 1 \Rightarrow 2P = 4 \Rightarrow P = 2$$

So **P = 2** ("same" padding, which is (F−1)/2 when stride is 1).

## Question 2 — CNN Architectures

**a. One important contribution of AlexNet**

AlexNet (2012) showed that a deep CNN trained on GPUs could win ImageNet by a huge margin (top-5 error ~15.3% vs ~26% for the next best). It popularized **ReLU activations** (much faster training than tanh/sigmoid), along with dropout and data augmentation to reduce overfitting. This is basically what started the deep learning boom in computer vision.

**b. Why VGGNet uses many small 3 × 3 filters**

Stacking small filters gives the same receptive field as one big filter but with fewer parameters and more non-linearity. Two 3×3 layers see a 5×5 region, three 3×3 layers see a 7×7 region. For C channels, three 3×3 layers use 3·(3²C²) = 27C² weights, while one 7×7 layer uses 49C². Also there is a ReLU after every layer, so the network can learn more complex functions. The design is also very simple and uniform.

**c. Main idea of the Inception module (GoogLeNet)**

Instead of picking one filter size, the Inception module runs **1×1, 3×3, 5×5 convolutions and a 3×3 max-pool in parallel** on the same input and concatenates the outputs along the channel dimension. So the network can capture features at multiple scales at the same time. To keep it cheap, **1×1 convolutions are used as bottlenecks** to reduce the number of channels before the expensive 3×3 and 5×5 convolutions.

**d. What problem residual learning (ResNet) addresses**

It addresses the **degradation problem** in very deep networks — when you just keep stacking layers, training accuracy actually gets worse, mainly because of vanishing gradients and difficult optimization. ResNet adds **skip (identity) connections** so a block learns a residual F(x) and outputs F(x) + x. If the extra layers aren't useful the block can just learn F(x) ≈ 0 (identity), and gradients can flow straight back through the shortcut. This made it possible to train networks with 100+ layers (ResNet-152).

## Question 3 — Representation Learning and Transfer Learning

**a. What is representation learning?**

Representation learning means the model **learns useful features from raw data automatically**, instead of us hand-designing them (like SIFT or HOG for images). A CNN learns a hierarchy of representations layer by layer, and the final representation makes the task (e.g. classification) easier.

**b. Why lower CNN layers learn different features than higher layers**

Lower layers have a small receptive field and only see tiny patches of the image, so they learn **generic, low-level features** like edges, color blobs and simple textures. Higher layers combine the outputs of earlier layers and see a much bigger part of the image, so they learn **more abstract, task-specific features** like object parts (eyes, wheels) and whole objects. That's why lower layers transfer well between datasets and the top layers are more specific to the original training task.

**c. What is transfer learning / fine-tuning?**

Transfer learning is taking a model already trained on a big dataset (like ImageNet) and reusing its learned weights for a new but related task. Fine-tuning is a type of transfer learning where we start from these pretrained weights and **keep training some (or all) of the layers on the new dataset**, usually with a small learning rate, so the features adapt to the new data.

**d. Why start with a pretrained CNN for a small dataset?**

A deep CNN has millions of parameters; training from scratch on a small dataset will badly overfit and probably won't converge to good features. A pretrained network already knows good general features (edges, textures, shapes) from millions of images, so we only need to learn a little bit on top. This gives better accuracy, needs much less data, and trains much faster / cheaper.

**e. Freezing vs fine-tuning a layer**

- **Freezing** a layer: its weights are fixed (`trainable = False`), no gradients are computed/applied for it, so it acts as a fixed feature extractor.
- **Fine-tuning** a layer: the layer stays trainable and its pretrained weights get updated by backprop on the new data (usually with a small learning rate) so they adapt to the new task.

---
# Part II — Programming Questions

## Question 1 — Implement Convolution from Scratch

Stride = 1, padding = 0, no built-in conv function — just two loops and an element-wise multiply + sum.
(Note: like in CNN libraries, this is technically cross-correlation, i.e. the filter is not flipped. Here the filter is symmetric so it doesn't matter anyway.)

In [ ]:
import numpy as np

# a. store input and filter as numpy arrays
input_img = np.array([[1, 1, 1, 0, 0],
                      [0, 1, 1, 1, 0],
                      [0, 0, 1, 1, 1],
                      [0, 0, 1, 1, 0],
                      [0, 1, 1, 0, 0]])

kernel = np.array([[1, 0, 1],
                   [0, 1, 0],
                   [1, 0, 1]])

print("Input:\n", input_img)
print("Filter:\n", kernel)

In [ ]:
def conv2d(image, kernel, stride=1, padding=0):
    # add zero padding around the image if needed
    if padding > 0:
        image = np.pad(image, padding, mode="constant", constant_values=0)

    n_h, n_w = image.shape
    f_h, f_w = kernel.shape

    # output size from the formula (N - F + 2P)/S + 1
    # (padding already added to image above so it's just (N - F)/S + 1 here)
    out_h = (n_h - f_h) // stride + 1
    out_w = (n_w - f_w) // stride + 1
    output = np.zeros((out_h, out_w), dtype=int)

    # b. slide the filter over the input
    for i in range(out_h):
        for j in range(out_w):
            r = i * stride
            c = j * stride
            patch = image[r:r + f_h, c:c + f_w]   # current 3x3 window
            # c. dot product = multiply element-wise and add everything up
            output[i, j] = np.sum(patch * kernel)
    return output


feature_map = conv2d(input_img, kernel, stride=1, padding=0)

# d. print the output feature map
print("Output feature map (stride=1, padding=0):")
print(feature_map)

# e. print the output shape
print("Output shape:", feature_map.shape)

In [ ]:
# quick check of one value by hand -> top-left window
top_left = input_img[0:3, 0:3]
print(top_left)
print("sum of top_left * kernel =", np.sum(top_left * kernel))   # 1+1+1+1 = 4 -> matches output[0,0]

In [ ]:
# f. what happens if stride = 2
out_s2 = conv2d(input_img, kernel, stride=2, padding=0)
print("Output with stride=2:")
print(out_s2)
print("Output shape:", out_s2.shape)

**f. Effect of changing stride from 1 to 2**

With stride 2 the filter jumps 2 pixels at a time instead of 1, so it visits fewer positions. Using the formula: (5 − 3 + 0)/2 + 1 = **2**, so the output shrinks from **3 × 3 to 2 × 2**. It only keeps the values at the corners of the stride-1 map (positions (0,0), (0,2), (2,0), (2,2)). In general a bigger stride downsamples the feature map (roughly halves height and width), reduces computation, but also loses some spatial detail because neighboring positions are skipped.

## Question 2 — Transfer Learning: Freeze vs. Fine-Tune

**Setup**
- **Pretrained model:** VGG16 trained on ImageNet (Keras, `include_top=False`)
- **Dataset:** 2 classes from CIFAR-10 — **cat vs dog** (binary classification). I use a subset to keep it runnable on CPU: 800 train images per class and 500 test images per class. Images are upscaled from 32×32 to 64×64.
- Images are loaded from the CIFAR-10 PNG mirror on GitHub (`YoongiKim/CIFAR-10-images`), only the cat/dog folders are downloaded.

**Experiments**
- **A – Feature extraction:** whole VGG16 base frozen, new classifier head trained only.
- **B – Fine-tuning:** same pretrained base, but the **last conv block (block5)** is unfrozen and trained together with the new head.

Both use the same head, same data, same batch size and **same number of epochs (5)**.

In [ ]:
import os, time, glob, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input

# fix seeds so results are (mostly) repeatable
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

In [ ]:
# download only the cat and dog folders of CIFAR-10 (sparse git checkout, ~20k small pngs)
DATA_DIR = "CIFAR-10-images"
if not os.path.exists(os.path.join(DATA_DIR, "train", "cat")):
    !git clone --depth 1 --filter=blob:none --sparse https://github.com/YoongiKim/CIFAR-10-images.git
    !cd CIFAR-10-images && git sparse-checkout set train/cat train/dog test/cat test/dog
else:
    print("dataset already downloaded")

In [ ]:
CLASSES = ["cat", "dog"]      # label 0 = cat, 1 = dog
IMG_SIZE = 64
TRAIN_PER_CLASS = 800
TEST_PER_CLASS = 500

def load_split(split, n_per_class):
    X, y = [], []
    for label, cls in enumerate(CLASSES):
        files = sorted(glob.glob(os.path.join(DATA_DIR, split, cls, "*.jpg")) +
                       glob.glob(os.path.join(DATA_DIR, split, cls, "*.png")))
        random.shuffle(files)
        for f in files[:n_per_class]:
            img = Image.open(f).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
            X.append(np.array(img))
            y.append(label)
    X = np.array(X, dtype="float32")
    y = np.array(y, dtype="float32")
    idx = np.random.permutation(len(X))   # shuffle so classes are mixed
    return X[idx], y[idx]

X_train, y_train = load_split("train", TRAIN_PER_CLASS)
X_test, y_test = load_split("test", TEST_PER_CLASS)

# VGG16 expects its own preprocessing (BGR + mean subtraction)
X_train = preprocess_input(X_train)
X_test = preprocess_input(X_test)

print("train:", X_train.shape, " class counts:", np.bincount(y_train.astype(int)))
print("test :", X_test.shape, " class counts:", np.bincount(y_test.astype(int)))

In [ ]:
# quick look at some of the images (reload a few raw ones just for display)
fig, axes = plt.subplots(1, 8, figsize=(14, 2))
for ax, cls_name in zip(axes, CLASSES * 4):
    f = random.choice(glob.glob(os.path.join(DATA_DIR, "train", cls_name, "*")))
    ax.imshow(Image.open(f))
    ax.set_title(cls_name)
    ax.axis("off")
plt.show()

In [ ]:
EPOCHS = 5
BATCH_SIZE = 32

def build_model(fine_tune=False):
    # load VGG16 with imagenet weights, no top (fully connected) layers
    base = VGG16(weights="imagenet", include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))

    if fine_tune:
        # Experiment B: freeze everything except the last conv block (block5)
        for layer in base.layers:
            layer.trainable = layer.name.startswith("block5")
    else:
        # Experiment A: freeze the whole conv base
        base.trainable = False

    # new classifier head (replaces the original 1000-class imagenet head)
    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)   # binary: cat vs dog
    model = keras.Model(inputs, outputs)

    # smaller learning rate when fine-tuning so we don't destroy the pretrained weights
    lr = 1e-4 if fine_tune else 1e-3
    model.compile(optimizer=keras.optimizers.Adam(lr),
                  loss="binary_crossentropy",
                  metrics=["accuracy"])
    return model


def count_params(model):
    trainable = int(sum(np.prod(w.shape) for w in model.trainable_weights))
    frozen = int(sum(np.prod(w.shape) for w in model.non_trainable_weights))
    return trainable, frozen


def run_experiment(name, fine_tune):
    tf.random.set_seed(SEED)
    model = build_model(fine_tune)
    trainable, frozen = count_params(model)
    print(f"\n===== {name} =====")
    print(f"trainable params: {trainable:,}   non-trainable params: {frozen:,}")

    start = time.time()
    history = model.fit(X_train, y_train,
                        validation_data=(X_test, y_test),
                        epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=2)
    train_time = time.time() - start

    test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
    print(f"training time: {train_time:.1f} s   test accuracy: {test_acc:.4f}")
    return {"name": name, "trainable": trainable, "time": train_time,
            "acc": test_acc, "history": history.history}

### Experiment A — Feature Extraction (frozen base)

In [ ]:
res_a = run_experiment("Frozen Feature Extractor", fine_tune=False)

### Experiment B — Fine-Tuning (block5 unfrozen)

In [ ]:
res_b = run_experiment("Fine-Tuned Network", fine_tune=True)

### Training loss plot

In [ ]:
epochs_range = range(1, EPOCHS + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

for res, style in [(res_a, "o-"), (res_b, "s-")]:
    ax1.plot(epochs_range, res["history"]["loss"], style, label=res["name"])
    ax2.plot(epochs_range, res["history"]["val_accuracy"], style, label=res["name"])

ax1.set_title("Training loss")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Binary cross-entropy")
ax1.legend(); ax1.grid(alpha=0.3)

ax2.set_title("Test accuracy per epoch")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Accuracy")
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("training_loss.png", dpi=120)
plt.show()

### Comparison table

In [ ]:
results = pd.DataFrame({
    "Method": [res_a["name"], res_b["name"]],
    "Trainable Parameters": [f"{res_a['trainable']:,}", f"{res_b['trainable']:,}"],
    "Training Time (s)": [round(res_a["time"], 1), round(res_b["time"], 1)],
    "Accuracy": [f"{res_a['acc']*100:.2f}%", f"{res_b['acc']*100:.2f}%"],
})
results

### g. Discussion

In [ ]:
# print the actual numbers so the discussion below can be checked against them
print(f"Frozen     : {res_a['trainable']:>10,} trainable params | {res_a['time']:.1f}s | acc {res_a['acc']*100:.2f}%")
print(f"Fine-tuned : {res_b['trainable']:>10,} trainable params | {res_b['time']:.1f}s | acc {res_b['acc']*100:.2f}%")
print(f"fine-tuning trains {res_b['trainable']/res_a['trainable']:.0f}x more parameters, "
      f"takes {res_b['time']/res_a['time']:.2f}x the time, "
      f"and changes accuracy by {(res_b['acc']-res_a['acc'])*100:+.2f} points")

**g. Discussion**

| Method | Trainable Parameters | Training Time | Accuracy |
|---|---|---|---|
| Frozen Feature Extractor | 131,585 | 15.3 s | 78.90% |
| Fine-Tuned Network | 7,211,009 | 15.4 s | 79.90% |

The fine-tuned network trains about 55 times more parameters than the frozen one (7.2M vs 131K), because the three conv layers of block5 are trainable in addition to the new classifier head. Even so, the training time was almost the same (15.3 s vs 15.4 s). Both models still run the full forward pass through all of VGG16, and that is most of the computation. The extra backward pass through block5 is cheap here: with 64×64 inputs, block5 only works on a 4×4 feature map, and the GPU handles that extra work easily. Fine-tuning gave a slightly higher accuracy (79.9% vs 78.9%). That makes sense, because block5 holds the most ImageNet-specific features, and letting it adapt helps it fit small, low-resolution CIFAR cat and dog images. The gain is small, though: only 1 point (10 test images out of 1000), which is close to run-to-run noise. Fine-tuning used a small learning rate (1e-4) for only 5 epochs, so block5 did not move far from its pretrained weights. With more epochs or larger images the gap would probably grow, but so would the risk of overfitting such a small training set. In general, freezing is cheaper and safer when data is limited, and fine-tuning can reach better accuracy when there is enough data and compute to adapt the higher layers.